# Expressive audiobook demo

Before the demo: set the runtime to **T4 GPU**, then run cells 1 to 4 once (about 3 minutes). Then, for each PDF, run the "Narrate a PDF" cell. The last cell is an optional web app.

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!pip -q uninstall -y torchao
!pip -q install -U peft bitsandbytes edge-tts pydub pymupdf gender-guesser

In [ ]:
SAMPLE_TEXT = """The storm had been circling the island since noon, and by evening the lighthouse shook with every wave.

"We should have left yesterday," Marta whispered, pulling the blanket tight around her shoulders.

Old Gregor did not answer. He stood at the window with his back to her, watching the lamp sweep across black water.

"The boat is gone, Marta," he said at last, very quietly. "The rope gave way an hour ago. I did not want to frighten you."

She stared at him. For a long moment the only sound was the wind clawing at the glass.

"Gone?" she said. "Then how will we send word to the mainland?"

"The radio still works," said young Pavel, bursting through the door with rain streaming from his coat. "I fixed it! Listen, they can hear us! They are sending a ship at dawn!"

Marta laughed, and then she was crying, and then laughing again, and Gregor, who had not smiled in a year, put his heavy hand on the boy's shoulder.

"Good lad," he murmured. "Good lad."

Outside, the storm roared on, but inside the little room it was suddenly warm. Somewhere far below, the sea kept its own counsel, patient and enormous, as though it had all the time in the world.
"""

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, time
BASE = "/content/drive/MyDrive/audiobook_genai"
CODE = f"{BASE}/code"
MODEL_PATH = f"{BASE}/adapter_final"
OUT_DIR = f"{BASE}/demo_output"
os.makedirs(OUT_DIR, exist_ok=True)
sys.path[:0] = [f"{CODE}/pipeline", f"{CODE}/dataset"]

for p in (f"{CODE}/pipeline/run.py", f"{CODE}/dataset/schema.py"):
    assert os.path.exists(p), f"missing: {p}"
assert any(os.path.exists(f"{MODEL_PATH}/{n}") for n in ("adapter_config.json", "config.json")), f"no model in {MODEL_PATH}"

from prepare import prepare
from annotate import label_units, load_generator
from narrate import narrate

t = time.time()
generate = load_generator(MODEL_PATH)
print(f"model ready in {time.time() - t:.0f} s")

In [ ]:
#@title Narrate a PDF
USE_SAMPLE = False  #@param {type:"boolean"}
START_PAGE = 1      #@param {type:"integer"}
END_PAGE = 3        #@param {type:"integer"}
MAX_UNITS = 25      #@param {type:"integer"}

import json, re
import pandas as pd
from google.colab import files
from IPython.display import Audio, display

if USE_SAMPLE:
    path, start, end = "/content/sample.txt", None, None
    open(path, "w", encoding="utf-8").write(SAMPLE_TEXT)
else:
    uploaded = files.upload()
    name = next(iter(uploaded))
    path, start, end = f"/content/input_{name}", START_PAGE, END_PAGE
    open(path, "wb").write(uploaded[name])

stem = re.sub(r"[^A-Za-z0-9_-]+", "_", os.path.splitext(os.path.basename(path))[0])
units = prepare(path, max_units=MAX_UNITS, start_page=start, end_page=end)
assert units, "No text found in the chosen pages"
mode = "script (NAME: lines)" if any("speaker_hint" in u for u in units) else "prose (quoted dialogue)"
print(f"{len(units)} units ({sum(u['kind'] == 'dialogue' for u in units)} dialogue), {mode} format")

t0 = time.time()
labels = label_units(units, generate)
print(f"labeled by the fine-tuned model in {time.time() - t0:.0f} s")
labels_path = f"{OUT_DIR}/{stem}_labels.json"
json.dump(labels, open(labels_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

pd.set_option("display.max_colwidth", 70)
display(pd.DataFrame([{"kind": r["kind"], "speaker": r["speaker"], "gender": r["gender"],
                       "emotion": r["emotion"], "intensity": r["intensity"], "pace": r["pace"],
                       "text": r["text"][:70]} for r in labels]))

out, flat = f"{OUT_DIR}/{stem}.mp3", f"{OUT_DIR}/{stem}_flat.mp3"
try:
    t0 = time.time()
    narrate(labels, out, f"{OUT_DIR}/cache")
    narrate(labels, flat, f"{OUT_DIR}/cache", flat=True)
    print(f"audio ready in {time.time() - t0:.0f} s")
    print("Single flat voice:")
    display(Audio(flat))
    print("Fine-tuned model, character voices, emotional delivery:")
    display(Audio(out))
except Exception as e:
    print("Speech step failed:", e)
    print("The labels are saved. Downloading them now so narration can run on the laptop.")
    files.download(labels_path)

In [ ]:
#@title Optional: launch the web app with the model already loaded
!pip -q install gradio
from pathlib import Path
sys.path.insert(0, CODE)
import app

app.SAMPLE = Path("/content/sample.txt")
app.SAMPLE.write_text(SAMPLE_TEXT, encoding="utf-8")
app.OUT_DIR = Path(OUT_DIR) / "app"
app.build_app(generate).queue().launch(share=True)